In [1]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU Available: []


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip -q /content/drive/MyDrive/Shapes/Shapes.zip -d /content/
!ls /content/

drive  sample_data  Shapes


In [4]:

import os

bad_images = [
    "img1141-W-4.png", "img1182-W-4.png", "img4051-W-4.png", "img4293-W-4.png",
    "img1955-W-3(1011).png", "img3930-W-3(1011).png",
    "img2489-T-5.png", "img3000-T-5.png", "img4294-T-5.png", "img2251-T-5.png",
    "img2302-T-0.png", "img1183-T-5.png", "img1706-T-5.png",
    "img3164-ST-3(11001).png", "img3054-ST-4(11101).png",
    "img1421-D-5.png", "img1549-D-5.png", "img1957-D-5.png",
    "img2303-D-0.png", "img2305-D-4(11011).png", "img4067-D-5.png",
    "img3053-D-5.png", "img2185-D-0.png",
    "img1954-O-5(101111).png", "img4292-O-6.png"
]

deleted_count = 0
for root, dirs, files in os.walk("/content/Shapes"):
    for file in files:
        if file in bad_images:
            os.remove(os.path.join(root, file))
            deleted_count += 1
print(f"Deleted: {deleted_count} images.")

Deleted: 25 images.


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_dir = "/content/Shapes"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False
)

train_gen = datagen.flow_from_directory(
    base_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

Found 3422 images belonging to 8 classes.
Found 852 images belonging to 8 classes.


In [6]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, DenseNet121
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json

# Use 5 epochs for CPU training
EPOCHS = 5
LR = 0.001

def build_and_train(model_name, base_model):
    print(f"\n{'='*50}\nTraining {model_name} on CPU...\n{'='*50}")

    # Keep base frozen for CPU speed
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(train_gen.num_classes, activation='softmax')
    ])

    model.compile(optimizer=optimizers.Adam(learning_rate=LR),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    save_path = f"/content/drive/MyDrive/Shapes/{model_name}_CPU_finetuned.keras"
    checkpoint = ModelCheckpoint(save_path, monitor='val_accuracy', save_best_only=True, verbose=1)

    history = model.fit(
        train_gen, validation_data=val_gen, epochs=EPOCHS,
        callbacks=[EarlyStopping(patience=2, restore_best_weights=True), checkpoint]
    )

    best_val_acc = max(history.history['val_accuracy'])
    print(f"--> {model_name} finished. Best Val Accuracy: {best_val_acc:.4f}")
    return best_val_acc

# 1. MobileNetV2
mobilenet_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
mobilenet_acc = build_and_train("MobileNetV2", mobilenet_base)

# 2. DenseNet121
densenet_base = DenseNet121(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
densenet_acc = build_and_train("DenseNet121", densenet_base)

print("\n" + "="*50)
print("PHASE 1 COMPLETE - CLASSIFIER COMPARISON (CPU)")
print("="*50)
print(f"ResNet-50 (Fine-Tuned GPU): 0.8650")
print(f"MobileNetV2 (CPU):          {mobilenet_acc:.4f}")
print(f"DenseNet121 (CPU):          {densenet_acc:.4f}")

# Save the comparison to Drive
results = {
    "ResNet50": 0.8650,
    "MobileNetV2": round(mobilenet_acc, 4),
    "DenseNet121": round(densenet_acc, 4)
}
with open("/content/drive/MyDrive/Shapes/classifier_comparison_CPU.json", "w") as f:
    json.dump(results, f, indent=4)
print("\nComparison saved to Drive.")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training MobileNetV2 on CPU...
Epoch 1/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7979 - loss: 0.6015
Epoch 1: val_accuracy improved from None to 0.97418, saving model to /content/drive/MyDrive/Shapes/MobileNetV2_CPU_finetuned.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Shapes/MobileNetV2_CPU_finetuned.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.9226 - loss: 0.2733 - val_accuracy: 0.9742 - val_loss: 0.1381
Epoch 2/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9792 - loss: 0.1087
Epoch 2: val_accuracy improved from 0.97418 to 0.98239, saving model to /content/drive/MyDrive/Shapes/MobileNetV2_CPU_finetuned.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Shapes/MobileNetV2_CPU_finetuned.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 138s 1s/step - accuracy: 0.9807 - loss: 0.1039 - val_accuracy: 0.9824 - val_loss: 0.1117
Epoch 3/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accu